## Introduction

This is a sample binary classification algorithm used to figure out if a patient has heart disease. In this example, we will upload sample data from Cleveland Heart Disease dataset taken from the UCI repository.  The dataset consists of 303 individuals data.  Please see data repository for column description and sample data.
https://archive.ics.uci.edu/ml/datasets/heart+Disease.  You can download the sample data, place the data in an S3 bucket, and execute the cells in this notebook to build and deploy you own model. 

### SDG 3: Good Health and Well-being

This project directly contributes to **Sustainable Development Goal 3 (SDG 3)** - ensuring healthy lives and promoting well-being for all at all ages. Specifically, this heart disease prediction system supports:

- **Target 3.4**: By 2030, reduce by one-third premature mortality from non-communicable diseases through prevention and treatment
- **Target 3.8**: Achieve universal health coverage, including access to quality essential healthcare services and access to safe, effective, quality, and affordable essential medicines and vaccines for all

The machine learning model enables early detection of cardiovascular disease, which is the leading cause of death globally. By providing predictive analytics for heart disease risk assessment, this technology helps healthcare providers:

1. **Early Intervention**: Identify high-risk patients before symptoms become severe
2. **Resource Optimization**: Prioritize healthcare resources for patients most at risk
3. **Preventive Care**: Enable proactive health management and lifestyle interventions
4. **Accessibility**: Provide affordable screening tools in resource-constrained settings

### AWS SageMaker Integration

This project leverages **AWS SageMaker** throughout the machine learning pipeline:

- **Data Processing**: SageMaker notebooks for data ingestion and preprocessing
- **Model Training**: SageMaker's Linear Learner algorithm for binary classification
- **Model Deployment**: SageMaker endpoints for real-time predictions
- **Scalability**: Managed infrastructure for training and inference
- **MLOps**: Integrated workflow for model lifecycle management

The rest of this tutorial walks you through using binary classification algorithm to predict heart disease using AWS SageMaker's fully managed machine learning platform.

## Prequisites and Preprocessing

### Permissions and environment variables

_This notebook was created and tested on an ml.m4.xlarge notebook instance._

Let's start by specifying:

- The S3 bucket and prefix that you want to use for training and model data.  This should be within the same region as the Notebook Instance, training, and hosting.  Once you have created your S3 bucket, specify the bucket name and prefix.
- The IAM role arn used to give training and hosting access to your data.

In [ ]:
# For local development, we'll use the local data file
# For AWS SageMaker deployment, uncomment and configure the S3 settings below:

# Enter bucket name
# bucket = '{ENTER_BUCKET_NAME}'
# prefix = 'sagemaker/heart'

# Enter data file name (e.g. heart.csv)
# data_key = 'heart.csv'
# data_location = 's3://{}/{}'.format(bucket, data_key)
 
# Define IAM role
import boto3
import re
import os
from sagemaker import get_execution_role

# For local development, use local file path
local_data_path = './heart-disease-predictor/src/main/resources/heart.csv'

# For SageMaker deployment, uncomment:
# role = get_execution_role()

# For local testing, we'll set role to None
role = None

### Data ingestion

For this demonstration, we'll use the local heart disease dataset. The AWS SageMaker implementation would typically read data from S3 buckets, but for local development and testing, we're using the included CSV file.

**AWS SageMaker Data Integration:**
- **S3 Storage**: In production, data is stored in S3 buckets for durability and scalability
- **Data Access**: SageMaker seamlessly integrates with S3 for training data ingestion
- **Data Formats**: Supports multiple formats including CSV, JSON, and RecordIO-protobuf
- **Data Security**: Encrypted storage and secure access through IAM roles

The code below reads the local data and prints out a sample of it.

In [ ]:
import pandas as pd
import json

# Read the data from local file for development
heart_data = pd.read_csv(local_data_path)

# For AWS SageMaker deployment, use:
# heart_data = pd.read_csv(data_location)

# Print out a sample of data
print("Dataset shape:", heart_data.shape)
print("\nFirst 5 rows:")
heart_data.head()

### Data conversion

**AWS SageMaker Data Processing:**
SageMaker algorithms have specific input and output requirements. This preprocessing step is crucial for optimal performance with SageMaker's built-in algorithms.

**Key SageMaker Features:**
- **RecordIO-protobuf Format**: Optimized format for high-performance training
- **Built-in Data Transformations**: SageMaker provides utilities for data format conversion
- **Distributed Processing**: Automatically handles data distribution across training instances
- **Memory Optimization**: Efficient data loading and processing for large datasets

The code below performs the following:

1. Convert the dataset to a numpy array of type float32 (required by SageMaker Linear Learner)
2. Create features matrix and labels vector for binary classification
3. Prepare data for SageMaker's RecordIO-protobuf format

**Data Structure:**
- Features: 13 medical parameters (age, sex, chest pain type, etc.)
- Target: Binary classification (0 = no heart disease, 1 = heart disease present)

This preprocessing ensures compatibility with SageMaker's Linear Learner algorithm and optimizes training performance.

In [ ]:
import numpy as np
vectors = np.array(heart_data).astype('float32')

#target column - value must be either 0 or 1
labels = vectors[:,13]
print ("label data is")
print (labels)


#drop the target column.  Use the features as part of the training data
training_data = vectors[:, :13]
print ("Training data is")
print (training_data)



Let's go ahead and upload the training data to S3

In [ ]:
import io
import os
import sagemaker.amazon.common as smac

buf = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf, training_data, labels)
buf.seek(0)

key = 'recordio-pb-data'
boto3.resource('s3').Bucket(bucket).Object(os.path.join(prefix, 'train', key)).upload_fileobj(buf)
s3_train_data = 's3://{}/{}/train/{}'.format(bucket, prefix, key)
print('uploaded training data location: {}'.format(s3_train_data))




## Training Artifacts
Once data is trained, it will be uploaded to the following location.

In [ ]:
output_location = 's3://{}/{}/output'.format(bucket, prefix)
print('training artifacts will be uploaded to: {}'.format(output_location))

## Training the linear model

### AWS SageMaker Training Infrastructure

Once we have the data preprocessed and available in the correct format for training, the next step is to actually train the model using the data. AWS SageMaker provides a fully managed training environment with:

**SageMaker Training Benefits:**
- **Managed Infrastructure**: Automatic provisioning and configuration of training instances
- **Distributed Training**: Built-in support for distributed training across multiple instances
- **Hyperparameter Optimization**: Automatic hyperparameter tuning with SageMaker Optimizer
- **Model Monitoring**: Built-in metrics and monitoring during training
- **Cost Optimization**: Pay only for the training time used
- **Scalability**: Easily scale from prototype to production workloads

**Linear Learner Algorithm:**
SageMaker's Linear Learner is optimized for:
- Large-scale linear models
- Binary and multi-class classification
- Regression tasks
- High-dimensional sparse data

Despite the dataset being small, provisioning hardware and loading the algorithm container take time upfront. In production scenarios, SageMaker's distributed training capabilities can significantly reduce training time for large datasets.

We will do a binary classification (patient either has heart disease or not), train the model on the specified compute (e.g. c4.xlarge), and we will sepcify the features or dimiensions in our training set.

In [ ]:
from sagemaker.amazon.amazon_estimator import get_image_uri
import sagemaker

container = get_image_uri(boto3.Session().region_name, 'linear-learner', "latest")

sess = sagemaker.Session()
linear = sagemaker.estimator.Estimator(container,
                                       role, 
                                       train_instance_count=1, 
                                       train_instance_type='ml.c4.xlarge',
                                       output_path=output_location,
                                       sagemaker_session=sess)
linear.set_hyperparameters(feature_dim=13,
                           predictor_type='binary_classifier',
                           mini_batch_size=100)

linear.fit({'train': s3_train_data})

## Set up hosting for the model

### AWS SageMaker Model Deployment

Now that we've trained our model, we can deploy it behind an Amazon SageMaker real-time hosted endpoint. This will allow us to make predictions (or inference) from the model dynamically.

**SageMaker Deployment Features:**
- **Real-time Endpoints**: Low-latency predictions for interactive applications
- **Auto Scaling**: Automatically scale endpoints based on traffic patterns
- **A/B Testing**: Deploy multiple model variants for comparison
- **Multi-model Endpoints**: Host multiple models on a single endpoint for cost efficiency
- **Shadow Testing**: Test new models alongside production models safely
- **Model Monitoring**: Track model performance and detect drift

**Deployment Options:**
- **Real-time Inference**: For low-latency, interactive applications
- **Batch Transform**: For offline, large-scale predictions
- **Serverless Inference**: For intermittent workloads with automatic scaling
- **Edge Deployment**: Deploy models to IoT devices using SageMaker Edge Manager

_Note, Amazon SageMaker allows you the flexibility of importing models trained elsewhere, as well as the choice of not importing models if the target of model creation is AWS Lambda, AWS Greengrass, Amazon Redshift, Amazon Athena, or other deployment target._

In [ ]:
heartdisease_predictor = linear.deploy(initial_instance_count=1,
                                 instance_type='ml.m4.xlarge')

## Validate the model for use

### AWS SageMaker Model Validation and Monitoring

Finally, we can now validate the model for use. We can pass HTTP POST requests to the endpoint to get back predictions. To make this easier, we'll again use the Amazon SageMaker Python SDK and specify how to serialize requests and deserialize responses that are specific to the algorithm.

**SageMaker Model Monitoring Features:**
- **Real-time Monitoring**: Track prediction quality and model performance
- **Data Drift Detection**: Automatically detect changes in input data distribution
- **Model Quality Monitoring**: Monitor accuracy, precision, recall, and other metrics
- **Alerting**: Set up alerts for performance degradation
- **Explainability**: Built-in feature importance and model explainability tools
- **A/B Testing**: Compare model performance with different versions

**Validation Metrics for Healthcare:**
- **Accuracy**: Overall prediction correctness
- **Sensitivity**: True positive rate (critical for disease detection)
- **Specificity**: True negative rate
- **Precision**: Positive predictive value
- **F1-Score**: Balance between precision and recall

For healthcare applications like heart disease prediction, high sensitivity is crucial to minimize false negatives.

In [ ]:
from sagemaker.predictor import csv_serializer, json_deserializer

heartdisease_predictor.content_type = 'text/csv'
heartdisease_predictor.serializer = csv_serializer
heartdisease_predictor.deserializer = json_deserializer

In [ ]:
print('Endpoint name: {}'.format(heartdisease_predictor.endpoint))

Let's try passing the following sample data for testing. This is a single record from the file

In [ ]:
vectors[5][0:13]

In [ ]:
result = heartdisease_predictor.predict(vectors[5][0:13])
print(result)

### (Optional) Delete the Endpoint

If you're ready to be done with this notebook, please run the delete_endpoint line in the cell below.  This will remove the hosted endpoint you created and avoid any charges from a stray instance being left on.

In [ ]:
## Summary: SDG 3 Impact and AWS SageMaker Integration

### Sustainable Development Goal 3 (Good Health and Well-being) Impact

This heart disease prediction system demonstrates how AI/ML can advance SDG 3 targets:

**Direct Contributions:**
- **Early Disease Detection**: Identifies at-risk patients before symptoms manifest
- **Preventive Healthcare**: Enables proactive interventions and lifestyle modifications
- **Healthcare Accessibility**: Provides affordable screening tools for underserved populations
- **Reduced Mortality**: Contributes to SDG 3.4 target of reducing premature deaths from NCDs

**Broader Healthcare Benefits:**
- **Resource Optimization**: Helps healthcare systems allocate resources efficiently
- **Cost Reduction**: Reduces expensive emergency interventions through early detection
- **Patient Empowerment**: Enables individuals to take control of their health
- **Public Health Planning**: Supports population-level health initiatives

### AWS SageMaker Complete ML Pipeline

This project showcases AWS SageMaker's comprehensive ML capabilities:

**1. Data Management**
- S3 integration for scalable data storage
- Built-in data preprocessing and transformation tools
- Support for multiple data formats (CSV, JSON, RecordIO)

**2. Model Development**
- Jupyter notebook environment for interactive development
- Built-in algorithms (Linear Learner) optimized for performance
- Hyperparameter tuning and optimization

**3. Training Infrastructure**
- Managed training instances with automatic scaling
- Distributed training for large datasets
- Cost-effective pay-as-you-go pricing

**4. Model Deployment**
- Real-time endpoints with auto-scaling
- Multiple deployment options (real-time, batch, serverless)
- A/B testing and shadow deployment capabilities

**5. Monitoring and Maintenance**
- Model performance monitoring
- Data drift detection
- Automated alerts and notifications

### Future Enhancements

**Technical Improvements:**
- Integration with electronic health records (EHR)
- Real-time patient monitoring systems
- Mobile health applications
- Telemedicine integration

**SDG 3 Expansion:**
- Multi-disease prediction models
- Population health analytics
- Health equity assessment tools
- Global health surveillance systems

This project demonstrates the powerful combination of AI/ML technology and cloud computing in addressing global health challenges and advancing sustainable development goals.